# Method 1: `mp.manager` to cache 

In [ ]:
def blocked_by_any_2(ij_pair, shared_data):
    """
    Determine if the direct path between two molecule centroids is obstructed by any other molecule.

    For a given pair of molecules (i, j), this function checks whether the straight line
    connecting their centroids is intersected ("blocked") by any other molecule in the system.
    The check is performed in two steps:
      1. Fast sphere rejection: For each candidate blocking molecule, if its centroid is not
         within its effective radius of the line segment, it is skipped.
      2. Ray-mesh intersection: If the sphere check passes, a ray-mesh intersection test is
         performed to determine if the mesh of the candidate molecule blocks the path.

    Periodic boundary conditions (PBC) are handled using the minimum-image convention.

    Parameters
    ----------
    i, j : int
        IDs of the two molecules to test for a direct connection.
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    ids : list[int]
        List of molecule IDs.

    Returns
    -------
    blocked : bool
        True if the path between i and j is blocked by any other molecule, False otherwise.
    """
    i, j = ij_pair
    
    # access the shared data through the proxy dictionary
    centroids = shared_data['centroids']
    radii = shared_data['radii']
    mol_meshes = shared_data['mol_meshes']
    neighbor_candidates = shared_data['neighbor_candidates']
    
    # Map molecule IDs to their index in ids
    ci, cj = centroids[i], centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return ij_pair
    direction = seg_vec / seg_len

    # Get candidate molecule IDs (not indices)
    cand_ids = [t[1] for t in neighbor_candidates if t[0] == i]

    for mol_k in cand_ids:
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return None
    return ij_pair

In [ ]:
import multiprocessing as mp 
# ... other necessary imports (trimesh, np, tqdm, partial) ...

def blocked_by_any_2(ij_pair, shared_data): # The worker function is defined correctly
    # ... (body of blocked_by_any_2 is fine as-is) ...
    # ...
    return ij_pair

# ---

def find_neighbors_2(centroids, radii, mol_meshes, box, neighbor_candidates, num_processes=None):
    
    total = len(neighbor_candidates)
    
    if num_processes is None: 
        num_processes = mp.cpu_count()
        
    # The Pool must be inside the Manager context to ensure shared_data is valid
    with mp.Manager() as manager:
        # Create a shared data dictionary to hold large objects
        shared_data = manager.dict()
        shared_data['centroids'] = centroids
        shared_data['radii'] = radii
        shared_data['mol_meshes'] = mol_meshes
        # neighbor_candidates does NOT need to be in shared_data, as it's iterable for the pool

        # The Pool processes will all share access to shared_data
        # worker_func needs to be a lambda or partial that takes (ij_pair) and passes it 
        # along with shared_data to the actual function.
        # Alternatively, use a partial that provides the shared_data.
        from functools import partial # You need to import this!
        worker_func = partial(blocked_by_any_2, shared_data=shared_data)

        with mp.Pool(processes=num_processes) as pool:
            tqdm_iterator = tqdm(
                # pool.imap maps worker_func to each element of neighbor_candidates
                # Each element (ij_pair) becomes the first argument (ij_pair) in worker_func
                # which then passes it to blocked_by_any_2
                pool.imap(worker_func, neighbor_candidates),
                total=total,
                desc=f'Finding unblocked neighbors with {num_processes} cores',
                colour='#7BC8F6'
            )
            
            neighbor_pairs = [pair for pair in tqdm_iterator if pair is not None]
            
    return neighbor_pairs

# Method 2: No-Cache

In [ ]:
def blocked_by_any_2(ij_pair, centroids, radii, mol_meshes, neighbor_candidates):
    """
    Determine if the direct path between two molecule centroids is obstructed by any other molecule.

    For a given pair of molecules (i, j), this function checks whether the straight line
    connecting their centroids is intersected ("blocked") by any other molecule in the system.
    The check is performed in two steps:
      1. Fast sphere rejection: For each candidate blocking molecule, if its centroid is not
         within its effective radius of the line segment, it is skipped.
      2. Ray-mesh intersection: If the sphere check passes, a ray-mesh intersection test is
         performed to determine if the mesh of the candidate molecule blocks the path.

    Periodic boundary conditions (PBC) are handled using the minimum-image convention.

    Parameters
    ----------
    i, j : int
        IDs of the two molecules to test for a direct connection.
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    ids : list[int]
        List of molecule IDs.

    Returns
    -------
    blocked : bool
        True if the path between i and j is blocked by any other molecule, False otherwise.
    """
    # Map molecule IDs to their index in ids
    i, j = ij_pair
    ci, cj = centroids[i], centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return False
    direction = seg_vec / seg_len

    # Get candidate molecule IDs (not indices)
    cand_ids = [t[1] for t in neighbor_candidates if t[0] == i]

    for mol_k in cand_ids:
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return True
    return False

In [ ]:
def find_neighbors_2(centroids, radii, mol_meshes, box, neighbor_candidates, num_processes=None):
    """
    Determine all unblocked neighbor pairs from a list of candidate molecule pairs.

    Parameters
    ----------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    box : np.ndarray
        Simulation box dimensions (in Å).
    neighbor_candidates : list[tuple[int, int]]
        List of candidate neighbor pairs (i, j) to test for blocking.

    Returns
    -------
    neighbor_pairs : list[tuple[int, int]]
        List of unblocked neighbor pairs (i, j).
    """
    neighbor_pairs = []
    ids = list(centroids.keys())
    total = len(neighbor_candidates)
    
    # Determine the number of processes
    if num_processes is None: 
        num_processes = mp.cpu_count()

    # The partial function only needs the shared proxy object
    worker_func = partial(blocked_by_any_2,
                          centroids=centroids,
                          radii=radii,
                          mol_meshes=mol_meshes,
                          neighbor_candidates=neighbor_candidates)
    
    with mp.Pool(processes=num_processes, initializer=worker_init, initargs=(centroids, radii, mol_meshes, neighbor_candidates)) as pool:
        tqdm_iterator = tqdm(
            pool.imap(worker_func, neighbor_candidates),
            total=total,
            desc=f'Finding unblocked neighbors with {num_processes} cores',
            colour='#7BC8F6'
        )
        
        for ij_pair, result in zip(neighbor_candidates, tqdm_iterator):
            if not result:  # If NOT blocked
                neighbor_pairs.append(ij_pair)  # Append the original pair
            
    return neighbor_pairs

In [ ]:
# results_2 = find_neighbors_2(centroids, radii, 'molecule_meshes', box, neighbor_candidates_sorted, num_processes=4)\

# Method 3: Global Shared Data

Using global namespace to access data. If this works then method 1 should work as well (same concept).

In [ ]:
import multiprocessing as mp
import numpy as np
from tqdm import tqdm

SHARED_DATA = {}

def worker_init(centroids, radii, mol_meshes, neighbor_candidates):
    """Initializes each worker process with the shared data."""
    global SHARED_DATA
    SHARED_DATA['centroids'] = centroids
    SHARED_DATA['radii'] = radii
    SHARED_DATA['mol_meshes'] = mol_meshes
    SHARED_DATA['neighbor_candidates'] = neighbor_candidates
    
    return None

In [ ]:
def blocked_by_any_2(ij_pair):
    """
    Determine if the direct path between two molecule centroids is obstructed by any other molecule.

    For a given pair of molecules (i, j), this function checks whether the straight line
    connecting their centroids is intersected ("blocked") by any other molecule in the system.
    The check is performed in two steps:
      1. Fast sphere rejection: For each candidate blocking molecule, if its centroid is not
         within its effective radius of the line segment, it is skipped.
      2. Ray-mesh intersection: If the sphere check passes, a ray-mesh intersection test is
         performed to determine if the mesh of the candidate molecule blocks the path.

    Periodic boundary conditions (PBC) are handled using the minimum-image convention.

    Parameters
    ----------
    i, j : int
        IDs of the two molecules to test for a direct connection.
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    ids : list[int]
        List of molecule IDs.

    Returns
    -------
    blocked : bool
        True if the path between i and j is blocked by any other molecule, False otherwise.
    """
    # import shared data
    global SHARED_DATA
    centroids = SHARED_DATA['centroids']
    radii = SHARED_DATA['radii']
    mol_meshes = SHARED_DATA['mol_meshes']
    neighbor_candidates = SHARED_DATA['neighbor_candidates']
    
    # Map molecule IDs to their index in ids
    i, j = ij_pair
    ci, cj = centroids[i], centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return False
    direction = seg_vec / seg_len

    # Get candidate molecule IDs (not indices)
    cand_ids = [t[1] for t in neighbor_candidates if t[0] == i]

    for mol_k in cand_ids:
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return True
    return False

In [ ]:
def find_neighbors_2(centroids, radii, mol_meshes, box, neighbor_candidates, num_processes=None):
    """
    Determine all unblocked neighbor pairs from a list of candidate molecule pairs.

    Parameters
    ----------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    box : np.ndarray
        Simulation box dimensions (in Å).
    neighbor_candidates : list[tuple[int, int]]
        List of candidate neighbor pairs (i, j) to test for blocking.

    Returns
    -------
    neighbor_pairs : list[tuple[int, int]]
        List of unblocked neighbor pairs (i, j).
    """
    neighbor_pairs = []
    total = len(neighbor_candidates)
    
    # Determine the number of processes
    if num_processes is None: 
        num_processes = mp.cpu_count()
    
    with mp.Pool(processes=num_processes, initializer=worker_init, initargs=(centroids, radii, mol_meshes, neighbor_candidates)) as pool:
        tqdm_iterator = tqdm(
            pool.imap(blocked_by_any_2, neighbor_candidates),
            total=total,
            desc=f'Finding unblocked neighbors with {num_processes} cores',
            colour='#7BC8F6'
        )
        
        neighbor_pairs = []
        for ij_pair, result in zip(neighbor_candidates, tqdm_iterator):
            if not result:  # If NOT blocked
                neighbor_pairs.append(ij_pair)  # Append the original pair
            
    return neighbor_pairs

In [ ]:
# results_2 = find_neighbors_2(centroids, radii, 'molecule_meshes', box, neighbor_candidates_sorted, num_processes=4)